In [1]:
# ============================================================
# 03_models_encoders.ipynb
# Adaptive Continuous Authentication — Modality Encoders
# ============================================================

"""
Goal:
  - Train per-modality encoders on session-level features
  - Supervised: user-ID classifier (MLP) → use penultimate layer as embedding
  - Unsupervised: autoencoder → use bottleneck as embedding
  - Export:
      checkpoints/
        keystroke_mlp.pt, keystroke_ae.pt
        mouse_mlp.pt, mouse_ae.pt
      embeddings/
        keystroke_embeddings.npy
        mouse_embeddings.npy
"""

# -----------------------------
# Imports
# -----------------------------
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

# -----------------------------
# Paths
# -----------------------------
DATA_DIR = Path("data")
CKPT_DIR = Path("checkpoints"); CKPT_DIR.mkdir(exist_ok=True)
EMB_DIR = Path("embeddings"); EMB_DIR.mkdir(exist_ok=True)

# -----------------------------
# Utils
# -----------------------------
def load_npz(path: Path):
    z = np.load(path, allow_pickle=True)
    X = z["features"].astype(np.float32)
    users = z["user_id"].astype(str)
    sessions = z["session_id"].astype(str)
    return X, users, sessions

def make_splits(users, test_size=0.2, seed=42):
    le = LabelEncoder()
    y = le.fit_transform(users)
    idx_train, idx_val = train_test_split(
        np.arange(len(y)), test_size=test_size, random_state=seed, stratify=y
    )
    return idx_train, idx_val, y, le

class NumpyDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.from_numpy(X).float()
        self.y = None if y is None else torch.from_numpy(y).long()
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        if self.y is None: return self.X[i]
        return self.X[i], self.y[i]

# -----------------------------
# Models
# -----------------------------
class MLPEncoder(nn.Module):
    """
    Simple classifier:
      in_dim -> 128 -> 64 (embedding) -> num_classes
    Use the 64-d layer as the embedding.
    """
    def __init__(self, in_dim, num_classes, dropout=0.1):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU()
        )
        self.head = nn.Linear(64, num_classes)

    def forward(self, x, return_embed=False):
        z = self.backbone(x)
        logits = self.head(z)
        if return_embed:
            return logits, z
        return logits

class AutoEncoder(nn.Module):
    """
    Symmetric autoencoder:
      in_dim -> 128 -> 64 (bottleneck) -> 128 -> in_dim
    """
    def __init__(self, in_dim, dropout=0.0):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU()
        )
        self.dec = nn.Sequential(
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, in_dim)
        )
    def forward(self, x, return_embed=False):
        z = self.enc(x)
        xhat = self.dec(z)
        if return_embed: return xhat, z
        return xhat

# -----------------------------
# Training loops
# -----------------------------
def train_classifier(X, y, in_dim, num_classes, device, ckpt_path, epochs=40, bs=64, lr=1e-3):
    ds_tr = NumpyDataset(X["train"], y["train"])
    ds_va = NumpyDataset(X["val"], y["val"])
    dl_tr = DataLoader(ds_tr, batch_size=bs, shuffle=True)
    dl_va = DataLoader(ds_va, batch_size=bs, shuffle=False)

    model = MLPEncoder(in_dim, num_classes).to(device)
    crit = nn.CrossEntropyLoss()
    opt = optim.AdamW(model.parameters(), lr=lr)
    best_acc, best_state = 0.0, None

    for ep in range(1, epochs+1):
        model.train()
        for xb, yb in dl_tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logits = model(xb)
            loss = crit(logits, yb)
            loss.backward()
            opt.step()

        # Val
        model.eval()
        preds, gts = [], []
        with torch.no_grad():
            for xb, yb in dl_va:
                xb = xb.to(device)
                logits = model(xb)
                pred = torch.argmax(logits, dim=1).cpu().numpy()
                preds.append(pred); gts.append(yb.numpy())
        acc = accuracy_score(np.concatenate(gts), np.concatenate(preds))
        if acc > best_acc:
            best_acc = acc
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        if ep % 5 == 0 or ep == 1:
            print(f"[MLP] epoch {ep:02d} val_acc={acc:.4f}")

    # Save best
    torch.save(best_state, ckpt_path)
    print(f"✅ Saved classifier → {ckpt_path} (best val_acc={best_acc:.4f})")
    # Load best into model
    model.load_state_dict(best_state)
    return model, best_acc

def train_autoencoder(X_all, in_dim, device, ckpt_path, epochs=60, bs=128, lr=1e-3):
    ds = NumpyDataset(X_all)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)

    model = AutoEncoder(in_dim).to(device)
    crit = nn.MSELoss()
    opt = optim.AdamW(model.parameters(), lr=lr)
    best_loss, best_state = float("inf"), None

    for ep in range(1, epochs+1):
        model.train()
        total = 0.0
        for xb in dl:
            xb = xb.to(device)
            opt.zero_grad()
            xhat = model(xb)
            loss = crit(xhat, xb)
            loss.backward()
            opt.step()
            total += loss.item() * xb.size(0)
        epoch_loss = total / len(ds)
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        if ep % 5 == 0 or ep == 1:
            print(f"[AE ] epoch {ep:02d} recon_loss={epoch_loss:.6f}")

    torch.save(best_state, ckpt_path)
    print(f"✅ Saved autoencoder → {ckpt_path} (best recon_loss={best_loss:.6f})")
    model.load_state_dict(best_state)
    return model, best_loss

def export_embeddings_classifier(model, X_all, device, out_path):
    model.eval()
    with torch.no_grad():
        xb = torch.from_numpy(X_all).float().to(device)
        _, Z = model(xb, return_embed=True)
        Z = Z.cpu().numpy().astype(np.float32)
    np.save(out_path, Z)
    print(f"💾 Saved classifier embeddings → {out_path} (shape={Z.shape})")
    return Z

def export_embeddings_autoencoder(model, X_all, device, out_path):
    model.eval()
    with torch.no_grad():
        xb = torch.from_numpy(X_all).float().to(device)
        _, Z = model(xb, return_embed=True)
        Z = Z.cpu().numpy().astype(np.float32)
    np.save(out_path, Z)
    print(f"💾 Saved AE embeddings → {out_path} (shape={Z.shape})")
    return Z

# -----------------------------
# Load data
# -----------------------------
Xk, users_k, sessions_k = load_npz(DATA_DIR / "features_keystroke.npz")
Xm, users_m, sessions_m = load_npz(DATA_DIR / "features_mouse.npz")

print("Keystroke:", Xk.shape, "users:", len(np.unique(users_k)))
print("Mouse    :", Xm.shape, "users:", len(np.unique(users_m)))

# -----------------------------
# Device
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# -----------------------------
# Train Keystroke encoders
# -----------------------------
idx_tr_k, idx_va_k, yk, le_k = make_splits(users_k, test_size=0.2)
Xk_tr, Xk_va = Xk[idx_tr_k], Xk[idx_va_k]
yk_tr, yk_va = yk[idx_tr_k], yk[idx_va_k]

keystroke_cls_path = CKPT_DIR / "keystroke_mlp.pt"
keystroke_ae_path  = CKPT_DIR / "keystroke_ae.pt"

model_k_cls, acc_k = train_classifier(
    X={"train": Xk_tr, "val": Xk_va},
    y={"train": yk_tr, "val": yk_va},
    in_dim=Xk.shape[1],
    num_classes=len(np.unique(yk)),
    device=device,
    ckpt_path=keystroke_cls_path,
    epochs=40, bs=64, lr=1e-3
)

model_k_ae, _ = train_autoencoder(
    X_all=Xk, in_dim=Xk.shape[1], device=device,
    ckpt_path=keystroke_ae_path, epochs=60, bs=128, lr=1e-3
)

# Export embeddings (all sessions)
Zk_cls = export_embeddings_classifier(model_k_cls, Xk, device, EMB_DIR / "keystroke_embeddings_cls.npy")
Zk_ae  = export_embeddings_autoencoder(model_k_ae, Xk, device, EMB_DIR / "keystroke_embeddings_ae.npy")

# -----------------------------
# Train Mouse encoders
# -----------------------------
idx_tr_m, idx_va_m, ym, le_m = make_splits(users_m, test_size=0.2)
Xm_tr, Xm_va = Xm[idx_tr_m], Xm[idx_va_m]
ym_tr, ym_va = ym[idx_tr_m], ym[idx_va_m]

mouse_cls_path = CKPT_DIR / "mouse_mlp.pt"
mouse_ae_path  = CKPT_DIR / "mouse_ae.pt"

model_m_cls, acc_m = train_classifier(
    X={"train": Xm_tr, "val": Xm_va},
    y={"train": ym_tr, "val": ym_va},
    in_dim=Xm.shape[1],
    num_classes=len(np.unique(ym)),
    device=device,
    ckpt_path=mouse_cls_path,
    epochs=40, bs=64, lr=1e-3
)

model_m_ae, _ = train_autoencoder(
    X_all=Xm, in_dim=Xm.shape[1], device=device,
    ckpt_path=mouse_ae_path, epochs=60, bs=128, lr=1e-3
)

# Export embeddings (all sessions)
Zm_cls = export_embeddings_classifier(model_m_cls, Xm, device, EMB_DIR / "mouse_embeddings_cls.npy")
Zm_ae  = export_embeddings_autoencoder(model_m_ae, Xm, device, EMB_DIR / "mouse_embeddings_ae.npy")

# -----------------------------
# Quick summary
# -----------------------------
print("\n==== Summary ====")
print(f"Keystroke MLP   val acc: {acc_k:.4f} | embeddings: {Zk_cls.shape}")
print(f"Keystroke AE    bottleneck embeddings: {Zk_ae.shape}")
print(f"Mouse MLP       val acc: {acc_m:.4f} | embeddings: {Zm_cls.shape}")
print(f"Mouse AE        bottleneck embeddings: {Zm_ae.shape}")
print("Checkpoints saved in:", CKPT_DIR.resolve())
print("Embeddings saved in :", EMB_DIR.resolve())

Keystroke: (408, 6) users: 51
Mouse    : (1676, 8) users: 10
Device: cpu


/Users/vkhawarey/Documents/git/moviecruncher/dataviz/multi_modal_hci_gen/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
/Users/vkhawarey/Documents/git/moviecruncher/dataviz/multi_modal_hci_gen/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
/Users/vkhawarey/Documents/git/moviecruncher/dataviz/multi_modal_hci_gen/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression pr

[MLP] epoch 01 val_acc=0.0244
[MLP] epoch 05 val_acc=0.0244
[MLP] epoch 10 val_acc=0.0244
[MLP] epoch 15 val_acc=0.0244
[MLP] epoch 20 val_acc=0.0488
[MLP] epoch 25 val_acc=0.0244
[MLP] epoch 30 val_acc=0.0488
[MLP] epoch 35 val_acc=0.0122
[MLP] epoch 40 val_acc=0.0610
✅ Saved classifier → checkpoints/keystroke_mlp.pt (best val_acc=0.0610)
[AE ] epoch 01 recon_loss=689.259434
[AE ] epoch 05 recon_loss=248.989259
[AE ] epoch 10 recon_loss=34.990175
[AE ] epoch 15 recon_loss=7.771363
[AE ] epoch 20 recon_loss=3.039916
[AE ] epoch 25 recon_loss=1.352480
[AE ] epoch 30 recon_loss=0.712812
[AE ] epoch 35 recon_loss=0.257057
[AE ] epoch 40 recon_loss=0.144070
[AE ] epoch 45 recon_loss=0.101136
[AE ] epoch 50 recon_loss=0.115651
[AE ] epoch 55 recon_loss=0.065172
[AE ] epoch 60 recon_loss=0.052704
✅ Saved autoencoder → checkpoints/keystroke_ae.pt (best recon_loss=0.052086)
💾 Saved classifier embeddings → embeddings/keystroke_embeddings_cls.npy (shape=(408, 64))
💾 Saved AE embeddings → embeddi